# Online Quiz: Sampling, Regression, and Nearest Neighbors
*Elements of Data Science* <br>
**Open Monday through Thursday.**

This notebook has two parts:

1. **Review and practice (not recorded).** A short recap of each topic, code you can run and change, and practice questions you can answer as many times as you like.
2. **The recorded quiz.** **12 questions** chosen from a pool of 30, with at least two from every topic. You get **one try per question**.
   - **Second chance:** Attempt 2 gives you **12 new questions** you have not seen. Both attempts are recorded.

|Topic|Review|Inferential Thinking|
|---|---|---|
|Population vs. sample|Section 1|[Ch. 10](https://inferentialthinking.com/chapters/10/Sampling_and_Empirical_Distributions.html)|
|Code: functions, loops, conditionals|Section 2|[Ch. 8](https://inferentialthinking.com/chapters/08/Functions_and_Tables.html), [Ch. 9](https://inferentialthinking.com/chapters/09/Randomness.html)|
|Sampling and simulation|Section 3|[Ch. 9.3](https://inferentialthinking.com/chapters/09/3/Simulation.html), [Ch. 10](https://inferentialthinking.com/chapters/10/Sampling_and_Empirical_Distributions.html)|
|Regression toolkit|Section 4|[Ch. 15](https://inferentialthinking.com/chapters/15/Prediction.html)|
|Nearest neighbors|Section 5|[Ch. 17](https://inferentialthinking.com/chapters/17/Classification.html)|

## ⚙️ Initialization — DO NOT EDIT
👁️ **Accessible view (optional):** set `ACCESSIBLE_VIEW = True` for larger text and colorblind-safe plots.

👉 **Run the cell below first** (Shift+Enter).

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
ACCESSIBLE_VIEW = False     # True = larger text and colorblind-safe plots

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..", "shared-public")))
if ACCESSIBLE_VIEW:
    from init_access import *
else:
    from init import *

In [ ]:
# 🔖 Quiz version — do not edit.
ver = '2026V110'

### 🧰 Standard imports

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
import json

Run the cell below to load the practice questions. Then enter your name.

In [ ]:
with open("quiz_practice.json") as f:
    practice_bank = json.load(f)

def practice(topic):
    "Show the practice questions for one topic (not recorded)."
    display_quiz([q for q in practice_bank if q['topic'] == topic], shuffle_answers=True)

In [ ]:
# Enter your name as a string
name = 'Jonathan'

## 1. Population vs. Sample
|Term|Meaning|Example|
|---|---|---|
|**Population**|the whole group we want to learn about|all 72 alcohols in `ROH_data`|
|**Sample**|the part of the population we actually look at|15 alcohols chosen at random|
|**Parameter**|a number that describes the **population**; fixed but usually unknown|the mean bp of all 72|
|**Statistic**|a number computed from a **sample**; changes from sample to sample|the mean bp of the 15|

- A **random sample** gives every individual a known chance of being chosen. A **convenience sample** (whoever is easy to reach) can be **biased**.
- **Law of averages:** a *large* random sample looks like the population it came from ([Ch. 10.1](https://inferentialthinking.com/chapters/10/1/Empirical_Distributions.html)).

Run the cell a few times. The **parameter** never changes; the **statistic** does.

In [ ]:
population = np.arange(1, 101)            # a population: the numbers 1 to 100
parameter = np.mean(population)

sample = np.random.choice(population, 10)
statistic = np.mean(sample)

print('Parameter (population mean):', parameter)
print('Statistic (sample mean):    ', statistic)

In [ ]:
practice('pop')

## 2. Code: Functions, Loops, and Conditionals
**Functions** package a computation so you can reuse it. Without `return`, a function gives back `None`.
```python
def standard_units(x):
    return (x - np.mean(x)) / np.std(x)
```
**Conditionals** run the *first* branch whose condition is `True`, then skip the rest ([Ch. 9.1](https://inferentialthinking.com/chapters/09/1/Conditional_Statements.html)).

**Loops** repeat a block once for each item; the pattern *start empty → append each time* collects results ([Ch. 9.2](https://inferentialthinking.com/chapters/09/2/Iteration.html)).

Predict the output **before** you run each cell.

In [ ]:
def acid_label(pka):
    if pka < 7:
        return 'stronger'
    elif pka < 10:
        return 'moderate'
    return 'weaker'

acid_label(4.8), acid_label(9.3), acid_label(15.7)

In [ ]:
labels = make_array()
for pka in make_array(4.8, 9.3, 15.7, 6.1):
    labels = np.append(labels, acid_label(pka))
labels

`np.where` labels a whole array in one step, with no loop. Lab 10 used it in `acid_class`:

In [ ]:
np.where(make_array(4.8, 9.3, 15.7, 6.1) < 7, 'stronger', 'weaker')

In [ ]:
practice('code')

## 3. Sampling and Simulation
|Tool|What it does|Replacement?|
|---|---|---|
|`np.random.choice(arr, n)`|`n` random items from an array|**with** (default)|
|`t.sample(n)`|`n` random rows of a Table|**with** (default)|
|`t.sample(n, with_replacement=False)`|`n` different rows|without|
|`np.random.seed(s)`|makes the "random" results repeatable|—|

**Simulation recipe** ([Ch. 9.3](https://inferentialthinking.com/chapters/09/3/Simulation.html)): (1) simulate one value of the statistic, (2) repeat it many times in a loop, collecting the values in an array, (3) draw a histogram: the **empirical distribution of the statistic**.

The cell below compares sample means for two sample sizes. Which histogram is narrower? Why?

In [ ]:
population = np.arange(1, 101)

def one_sample_mean(size):
    return np.mean(np.random.choice(population, size))

means_25 = make_array()
means_400 = make_array()
for i in np.arange(1000):
    means_25 = np.append(means_25, one_sample_mean(25))
    means_400 = np.append(means_400, one_sample_mean(400))

Table().with_columns('sample size 25', means_25, 'sample size 400', means_400).hist(bins=np.arange(30, 71, 1))
plt.xlabel('sample mean')
plt.show()

Proportions come from `np.mean` of a True/False array:

In [ ]:
flips = np.random.choice(make_array('H', 'T'), 10000)
np.mean(flips == 'H')

In [ ]:
practice('sampling')

## 4. The Regression Toolkit
|Step|Formula|
|---|---|
|standard units|$z = \dfrac{x - \text{mean}}{\text{SD}}$|
|correlation|$r = \text{mean of }(z_x \times z_y)$, between −1 and 1|
|slope|$r \times \dfrac{\text{SD of }y}{\text{SD of }x}$|
|intercept|$\text{mean of }y - \text{slope} \times \text{mean of }x$|
|residual|actual − predicted|

- $r$ measures **linear** association only. A strong curved pattern can have $r \approx 0$, so always look at the scatter plot ([Ch. 15.1](https://inferentialthinking.com/chapters/15/1/Correlation.html)).
- The regression line is the line with the smallest RMSE ([Ch. 15.3](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)).

Here is the toolbox from Lab 8, applied to a small made-up data set:

In [ ]:
def standard_units(any_numbers):
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))



In [ ]:
demo = Table().with_columns('MW', make_array(32, 46, 60, 74, 88, 102),
                            'bp', make_array(338, 351, 370, 391, 411, 430))
m = slope(demo, 'MW', 'bp')
b = intercept(demo, 'MW', 'bp')
print(f'r = {correlation(demo, "MW", "bp"):.3f},  slope = {m:.2f} K per g/mol,  intercept = {b:.1f} K')

demo = demo.with_columns('predicted', m * demo.column('MW') + b)
demo.with_columns('residual', demo.column('bp') - demo.column('predicted'))

In [ ]:
practice('regression')

## 5. Nearest Neighbors
1. **Distance:** $d = \sqrt{\sum (\text{difference in each feature})^2}$ ([Ch. 17.3](https://inferentialthinking.com/chapters/17/3/Rows_of_Tables.html))
2. **Find** the $k$ training rows closest to the new example.
3. **Predict:** the **average** of their targets (regression), or the **majority vote** of their classes (classification, [Ch. 17.4](https://inferentialthinking.com/chapters/17/4/Implementing_the_Classifier.html)).

|Idea|Remember|
|---|---|
|Scaling|Convert features to standard units, using the **training** mean and SD, so no feature dominates|
|Training / validation / test|neighbors / choose k and features / report once at the end|
|RMSE|typical size of a prediction error, in the units of the target|
|Accuracy|proportion of classifications that are correct: `np.mean(predicted == actual)`|

Change `k` below and see how the prediction for the test point changes.

In [ ]:
def distance(pt1, pt2):
    return np.sqrt(np.sum((pt1 - pt2) ** 2))

toy = Table().with_columns('x', make_array(3, 8, 5, 6, 1, 9, 8),
                           'y', make_array(4, 5, 7, 4, 6, 9, 4),
                           'target', make_array(10, 30, 22, 18, 12, 40, 28))
test_point = make_array(4, 8)

dists = make_array()
for row in toy.select('x', 'y').rows:
    dists = np.append(dists, distance(np.array(row), test_point))

k = 3
neighbors = toy.with_columns('distance', dists).sort('distance').take(np.arange(k))
print(f'kNN prediction with k = {k}:', np.mean(neighbors.column('target')))
neighbors

In [ ]:
practice('knn')

---
## 📝 Recorded Quiz
- **12 questions** drawn from a pool of 30, at least two from each topic. **One try per question.**
- Your questions are chosen for **you** and stay the same if you re-run the cell, so re-running won't give you easier ones.
- **Attempt 2 (second chance)** gives you **12 new questions** that weren't in Attempt 1. Both attempts are recorded.
- The quiz is open **Monday through Thursday**.

Finish the review above first. When you are ready, run **Attempt 1**.

In [ ]:
# Quiz settings — do not edit
from pool_quiz import open_pool_quiz
QUIZ_ID = 'OnlineQuiz'
OPENS, CLOSES = '2026-11-15 00:00', '2026-11-19 23:59'

### Attempt 1

In [ ]:
quiz_result_1 = open_pool_quiz('/home/jovyan/data/Fall 2026/shared-public/quiz_pool.json', name, user, QUIZ_ID, attempt=1, opens=OPENS, closes=CLOSES)

### Attempt 2: Second chance (new questions)
Only run this after you finish Attempt 1. Before you start, go back to the review sections for any topic that gave you trouble.

In [ ]:
quiz_result_2 = open_pool_quiz('/home/jovyan/data/Fall 2026/shared-public/quiz_pool.json', name, user, QUIZ_ID, attempt=2, opens=OPENS, closes=CLOSES)